# Load custom quadruped robot env

In this tutorial we will see how to use the `MuJoCo/Ant-v5` framework to create a quadruped walking env, using a model file (ending in .`xml`) without having to create a new class

**Steps::**
1. Get MJCF model file of your robot
    - create your own model
    - Find a ready-made model
2. Load the model with the `xml_file` argument
3. Tweak the env parameters to get the desired behavior
    - Tweak the environment simulation parameters
    - Tweak the env termination parameters
    - Tweak the env reward parameters
    - Tweak the env observation parameters
4. Train an agent to move you robot

## Step1: Download a Robot Model

In this tutorial we will load the `Unitree Go` robot from the excellent `MuJoCo Menagerie` robot model collection.

![img](https://github.com/google-deepmind/mujoco_menagerie/blob/main/unitree_go1/go1.png?raw=true)

`Go1` is a quadruped robot, controlling it to move is a significant learning problem, much harder than the `Gymnasium/MuJoCo/Ant` env.


## Step2: Load the Model

To load the model, all we have to do is use the `xml_file` argument with the `Ant-v5` framework

In [2]:
import gymnasium
import numpy as np
env = gymnasium.make('Ant-v5', xml_file='../../mujoco_menagerie/unitree_go1/scene.xml')

Although this is enough to load the model, we will need to tweak some env parameters to get the desired behavior for our env, for now we will also explicitly set the simulation, termination, reward and observation arguments, which we will tweak in the next step.

In [5]:
env = gymnasium.make(
    'Ant-v5',
    xml_file='../../mujoco_menagerie/unitree_go1/scene.xml',
    forward_reward_weight=0,
    ctrl_cost_weight=0,
    contact_cost_weight=0,
    healthy_reward=0,
    main_body=1,
    healthy_z_range=(0, np.inf),
    include_cfrc_ext_in_observation=True,
    exclude_current_positions_from_observation=False,
    reset_noise_scale=0,
    frame_skip=1,
    max_episode_steps=1000,
)

## Step 3: Tweaking the Env simulation Parameters

The arguments of interest are `frame_skip, reset_noise_scale, max_episode_steps`

We want to tweak the `frame_skip`parameter to get `dt` to an acceptable value.

Reminder: $$dt=frame_skipt * model.opt.timestep $$, where `model.opt.timestep` is the integrator time step selected in the MJCF model file

The `Go1` model we are using has an integrator timestep of `0.002`, so by selecting `frame_skip=25` we can set the value of `dt` to `0.05s`.

To avoid overfitting the policy, `reset_noise_scale` should be set to a value appropriate to the size of the robot, we want the value to be as large as possible without the initial distribution of states being invalid, for `Go1` we choose a value of `0.1`

And `max_episode_steps` determines the number of steps per episode before `truncation`, here we set it to 1000 to be consistent with the based `Gymnasium/MuJoCo` env, but if you need something higher you can set it so.